In [ ]:
%pip install -q langchain langchain-huggingface langchain-community langchain-core langchain-text-splitters bitsandbytes docx2txt langchain-chroma

In [ ]:
import os

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

loader = Docx2txtLoader('./tax_with_markdown.docx')
documents_list = loader.load_and_split(text_splitter)   


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='intfloat/multilingual-e5-large-instruct')

In [ ]:
from langchain_chroma import Chroma

collection_name = 'chroma_tax'

database = Chroma.from_documents(
    documents=documents_list,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory="./chorma_huggungface"    
)

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

chat_model = HuggingFacePipeline.from_model_id(
    model_id="LGAI-EXAONE/EXAONE-3.0-7.8B-Instruct",
    task="text-generation",
    pipeline_kwargs=dict(
        max_new_tokens=1024,
        do_sample=False,
        repetition_penalty=1.03
    ),
    model_kwargs={
        "quantization_config": quantization_config,
        "trust_remote_code": True,
        },
    
)

In [ ]:
llm = ChatHuggingFace(llm=chat_model)

In [ ]:
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_classic.chains import create_retrieval_chain
from langchain_classic import hub

retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")
retriever = database.as_retriever(search_kwargs={"k": 1})


In [ ]:
query = "연봉 5천만원인 거주자의 소득세는?"


retrieve_docs = retriever.invoke(query)

In [ ]:
retrieve_docs

In [ ]:
combine_docs_chain = create_stuff_documents_chain(
    llm, retrieval_qa_chat_prompt
)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [ ]:
ai_message = retrieval_chain.invoke({"input": query})

In [ ]:
ai_message

In [ ]:
test_ai_message = llm.invoke('인프런에는 어떤 강의가 있나요?')

In [ ]:
test_ai_message